##### __Author:__ Bram Van de Sande

__Date:__ 1 MAY 2018

__Outline:__ Tutorial on how to run the full SCENIC pipeline using the python implementation (i.e. pyscenic).

For this notebook 3005 single cell transcriptomes taken from the mouse brain (somatosensory cortex and hippocampal regions) are used as an example.

> A. Zeisel, A. B. M͡oz-Manchado, S. Codeluppi, P. Lönnerberg, G. L. Manno, A. Juréus, S. Marques, H. Munguba, L. He, C. Betsholtz, C. Rolny, G. Castelo-Branco, J. Hjerling-Leffler, and S. Linnarsson, “Cell types in the mouse cortex and hippocampus revealed by single-cell RNA-seq,” Science, vol. 347, no. 6226, pp. 1138–1142, Mar. 2015.

In [1]:
import pandas as pd
import numpy as np
import os, glob
import pickle
import pyarrow.feather as feather

from arboreto.utils import load_tf_names
from arboreto.algo import grnboost2

from pyscenic.rnkdb import FeatherRankingDatabase as RankingDatabase
#from ctxcore.rnkdb import FeatherRankingDatabase as RankingDatabase
from pyscenic.utils import modules_from_adjacencies
from pyscenic.prune import prune2df, df2regulons
from pyscenic.aucell import aucell

#import seaborn as sns

In [54]:
DATA_FOLDER = "/home/melanie/Documents/Projects/CLL_GRN_conditions/Data/"
RESOURCES_FOLDER="/home/melanie/Documents/Projects/CRCT/pySCENIC/resources/"
DATABASE_FOLDER = "/home/melanie/Documents/Projects/CRCT/pySCENIC/cisTarget_databases/"
#RESULT_FOLDER = "/home/melanie/Documents/Projects/CLL_GRN_conditions/Results/

DATABASES_GLOB = os.path.join(RESOURCES_FOLDER, "hg19-*7species.mc9nr.feather")
MOTIF_ANNOTATIONS_FNAME = os.path.join(RESOURCES_FOLDER, "motifs-v9-nr.hgnc-m0.001-o0.0.tbl")

MM_TFS_FNAME = os.path.join(RESOURCES_FOLDER, 'allTFs_hg38.txt')
SC_EXP_FNAME = os.path.join(DATA_FOLDER, "TPM_normalised_WO.csv")

ADJACENCIES_FNAME = os.path.join(DATA_FOLDER, "adjacencies_CLL_3cond.tsv")
MODULES_FNAME = os.path.join(DATA_FOLDER, "modules_CLL_WO.p")
MOTIFS_FNAME = os.path.join(DATA_FOLDER, "motifs_CLL_WO.csv")
REGULONS_FNAME = os.path.join(DATA_FOLDER, "regulons_CLL_WO.p")
AUCell_FNAME = os.path.join(DATA_FOLDER, "AUcell_CLL_WO.csv")

N_SAMPLES = 500

# Preliminary work

### Load the expression matrix

The scRNA-Seq data is downloaded from GEO: https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE60361 .

In [55]:
ex_matrix = pd.read_csv(SC_EXP_FNAME, sep=',', header=0, index_col=0).T

Remove duplicate genes.

In [56]:
ex_matrix.head()

,MT-ND1,MT-ND2,MT-CO1,MT-CO2,MT-ATP8,MT-ATP6,MT-CO3,MT-ND3,MT-ND4L,MT-ND4,...,ATP8B2,S100A13,TADA1,SNAP47,TSPAN1,AHCYL1,JMJD4,TNFRSF14,DAP3,YY1AP1
ABC_J1_WO_S34.genes.results.TPM,10.786023,11.228523,12.447537,12.337822,12.751310,12.138428,12.528139,12.160930,11.739387,11.717433,...,4.237737,4.931618,4.466051,5.786355,2.334981,5.071817,4.718979,6.921872,6.598433,5.889289
ABC_J11_WO_S43.genes.results.TPM,10.413441,11.114958,12.426753,12.228306,12.481302,11.894621,12.430980,11.734677,11.478407,11.508584,...,4.201643,5.989565,4.372135,5.562121,2.334981,5.044968,4.372135,6.694064,6.483852,6.130061
ABC_J8_WO_S40.genes.results.TPM,10.693490,10.969280,12.566610,12.201668,11.963921,11.862603,12.357433,11.404577,11.191131,11.640088,...,4.291293,6.241985,4.382547,5.688026,2.334981,5.057543,4.762055,7.085080,6.691389,6.171415
DIC_J1_WO_S10.genes.results.TPM,11.310897,11.625897,12.974404,11.807205,12.397620,11.959224,12.843655,11.938337,12.018170,12.091954,...,4.390166,3.314856,4.329899,5.917803,2.334981,5.295326,4.447733,6.245218,5.993256,5.858466
DIC_J11_WO_S19.genes.results.TPM,12.491495,12.558110,13.121756,12.464393,12.718003,12.269875,12.442402,12.236425,12.156489,12.199229,...,4.295081,3.841548,4.295081,4.766459,2.334981,5.213330,4.386464,7.259417,6.050953,5.941434


In [57]:
ex_matrix.shape

(15, 16188)

### Derive list of Transcription Factors(TF) for _Mus musculus_

List of known TFs for Mm was prepared (cf. notebook).

In [10]:
tf_names = load_tf_names(MM_TFS_FNAME)

In [11]:
len(tf_names)

1892

In [12]:
overlap = set(tf_names) & set(ex_matrix.columns)
len(overlap)

1613

### Load ranking databases

In [13]:
db_fnames = glob.glob(DATABASES_GLOB)
def name(fname):
    return os.path.splitext(os.path.basename(fname))[0]
dbs = [RankingDatabase(fname=fname, name=name(fname)) for fname in db_fnames]

In [14]:
dbs

[FeatherRankingDatabase(name="hg19-tss-centered-5kb-7species.mc9nr"),
 FeatherRankingDatabase(name="hg19-500bp-upstream-7species.mc9nr")]

In [15]:
df = feather.read_table(
    "/home/melanie/Documents/Projects/CRCT/pySCENIC/resources/New_database_files/hg19-500bp-upstream-7species.mc9nr.genes_vs_motifs.rankings.feather"
).to_pandas()

print(df.columns[:10])
print(df.head())

Index(['A1BG', 'A1BG-AS', 'A1CF', 'A2LD1', 'A2M', 'A2ML1', 'A4GALT', 'A4GNT',
       'AA06', 'AAA1'],
      dtype='object')
    A1BG  A1BG-AS   A1CF  A2LD1   A2M  A2ML1  A4GALT  A4GNT   AA06   AAA1  \
0  19697    12964  15692  19436  9944  20837   19169  22064  20913  11448   
1  12392    15392  11042   8906  4093  21218   11777  11738   5668   3664   
2  19036    10983   3630  18739  8067  20452    7040   2189  20567   9070   
3  12094     4900   4571   5666  1099  12104   15227   2627    595  18888   
4  15126    19903    614  10992  2256   1526    2633   3169   8387   8936   

   ...   ZXDB   ZXDC  ZYG11A  ZYG11B    ZYX  ZZEF1   ZZZ3  psiTPTE22   tAKR  \
0  ...  15077   3072   12674   12673  21064   7338   2574      20395   8755   
1  ...   3136   4530    9557   15142   6474  20279   5353       5316  14058   
2  ...  13402   3687   10583    1337    306  18888  19421      19903  10844   
3  ...   8345  20817   18783   13110  18138  20500  16234       7036  20370   
4  ...    151  114

# Phase I: Inference of co-expression modules

In the initial phase of the pySCENIC pipeline the single cell expression profiles are used to infer co-expression modules from.

### Run GRNboost from `arboreto` to infer co-expression modules

The arboreto package is used for this phase of the pipeline.

In [15]:
adjancencies = grnboost2(expression_data=ex_matrix, tf_names=tf_names, verbose=True)

preparing dask client
parsing input
creating dask graph
11 partitions
computing dask graph


/home/melanie/miniconda3/envs/pyscenic/lib/python3.10/site-packages/distributed/client.py:3169: UserWarning: Sending large graph of size 11.47 MiB.
This may cause some slowdown.
Consider scattering data ahead of time and using futures.
  warnings.warn(


shutting down client and local cluster
finished


In [16]:
adjancencies.head()

,TF,target,importance
340,HOXC4,MB21D2,132.133472
1593,LRRFIP1,MOGAT2,121.632641
1168,ASCC1,NLRP2,117.498796
191,FOXO4,NTN4,112.042872
780,HOXB7,LDOC1,109.566635


In [17]:
len(adjancencies)

10305437

In [18]:
adjancencies.to_csv(ADJACENCIES_FNAME, index=False, sep='\t')
# adjancencies = pd.read_csv("/home/melanie/Documents/Projects/DigPhat/Data/ccRCC_data_CEA/epithelial_data/", sep='\t')

In [58]:
adjancencies = pd.read_csv("/home/melanie/Documents/Projects/CLL_GRN_conditions/Results/dynGENIE3/GRN_dynGENIE3_WO.csv")
adjancencies.head()

,TF,target,importance
0,SIRT6,RALA,0.033856
1,JRKL,NSMCE3,0.033751
2,ZNF875,RALGAPB,0.033286
3,MBD1,RPAP1,0.033166
4,NFYA,ERCC5,0.032101


In [59]:
len(adjancencies)

1048575

### Derive potential regulomes from these co-expression modules

Regulons are derived from adjacencies based on three methods.

The first method to create the TF-modules is to select the best targets for each transcription factor:

1. Targets with importance > the 50th percentile.
2. Targets with importance > the 75th percentile
3. Targets with importance > the 90th percentile.

The second method is to select the top targets for a given TF:

1. Top 50 targets (targets with highest weight)

The alternative way to create the TF-modules is to select the best regulators for each gene (this is actually how GENIE3 internally works). Then, these targets can be assigned back to each TF to form the TF-modules. In this way we will create three more gene-sets:

1. Targets for which the TF is within its top 5 regulators
2. Targets for which the TF is within its top 10 regulators
3. Targets for which the TF is within its top 50 regulators

A distinction is made between modules which contain targets that are being activated and genes that are being repressed. Relationship between TF and its target, i.e. activator or repressor, is derived using the original expression profiles. The Pearson product-moment correlation coefficient is used to derive this information.

In addition, the transcription factor is added to the module and modules that have less than 20 genes are removed.

In [60]:
modules = list(modules_from_adjacencies(adjancencies, ex_matrix))


2026-06-24 16:35:24,190 - pyscenic.utils - INFO - Calculating Pearson correlations.

2026-06-24 16:35:24,277 - pyscenic.utils - WARNING - Note on correlation calculation: the default behaviour for calculating the correlations has changed after pySCENIC verion 0.9.16. Previously, the default was to calculate the correlation between a TF and target gene using only cells with non-zero expression values (mask_dropouts=True). The current default is now to use all cells to match the behavior of the R verision of SCENIC. The original settings can be retained by setting 'rho_mask_dropouts=True' in the modules_from_adjacencies function, or '--mask_dropouts' from the CLI.
	Dropout masking is currently set to [False].

2026-06-24 16:35:28,688 - pyscenic.utils - INFO - Creating modules.


In [61]:
with open(MODULES_FNAME, 'wb') as f:
    pickle.dump(modules, f)

with open(MODULES_FNAME, 'rb') as f:
    modules = pickle.load(f)

# Phase II: Prune modules for targets with cis regulatory footprints (aka RcisTarget)

In [62]:
def main():

    df = prune2df(
                dbs,
                modules,
                MOTIF_ANNOTATIONS_FNAME,
                client_or_address="custom_multiprocessing",
                num_workers=4,
                module_chunksize=10,
                weighted_recovery=False,
                rank_threshold=2000,
                auc_threshold=0.01
        )

    df.to_csv(MOTIFS_FNAME)

    regulons = df2regulons(df)
    with open(REGULONS_FNAME, 'wb') as f:
        pickle.dump(regulons, f)

    # AUCell (already memory-friendly if num_workers=1)
    auc_mtx = aucell(
        ex_matrix,
        regulons,
        num_workers=1
    )
    auc_mtx.to_csv(AUCell_FNAME, sep="\t")


if __name__ == "__main__":
    import multiprocessing as mp
    mp.set_start_method("fork", force=True)
    main()



2026-06-24 16:36:27,876 - pyscenic.prune - INFO - Using 4 workers.

2026-06-24 16:36:27,876 - pyscenic.prune - INFO - Using 4 workers.

2026-06-24 16:36:28,621 - pyscenic.prune - INFO - Worker hg19-tss-centered-5kb-7species.mc9nr(1): database loaded in memory.

2026-06-24 16:36:28,621 - pyscenic.prune - INFO - Worker hg19-tss-centered-5kb-7species.mc9nr(1): database loaded in memory.

2026-06-24 16:36:28,721 - pyscenic.prune - INFO - Worker hg19-tss-centered-5kb-7species.mc9nr(2): database loaded in memory.

2026-06-24 16:36:28,721 - pyscenic.prune - INFO - Worker hg19-tss-centered-5kb-7species.mc9nr(2): database loaded in memory.

2026-06-24 16:36:28,848 - pyscenic.prune - INFO - Worker hg19-500bp-upstream-7species.mc9nr(1): database loaded in memory.

2026-06-24 16:36:28,848 - pyscenic.prune - INFO - Worker hg19-500bp-upstream-7species.mc9nr(1): database loaded in memory.

2026-06-24 16:36:28,938 - pyscenic.prune - INFO - Worker hg19-500bp-upstream-7species.mc9nr(2): database loaded


2026-06-24 16:36:55,053 - pyscenic.transform - WARNING - Less than 80% of the genes in ENO1 could be mapped to hg19-500bp-upstream-7species.mc9nr. Skipping this module.

2026-06-24 16:36:56,045 - pyscenic.transform - WARNING - Less than 80% of the genes in ELK1 could be mapped to hg19-tss-centered-5kb-7species.mc9nr. Skipping this module.

2026-06-24 16:36:56,551 - pyscenic.transform - WARNING - Less than 80% of the genes in ELK3 could be mapped to hg19-tss-centered-5kb-7species.mc9nr. Skipping this module.

2026-06-24 16:36:56,601 - pyscenic.transform - WARNING - Less than 80% of the genes in ENO1 could be mapped to hg19-tss-centered-5kb-7species.mc9nr. Skipping this module.

2026-06-24 16:37:00,200 - pyscenic.transform - WARNING - Less than 80% of the genes in FIP1L1 could be mapped to hg19-500bp-upstream-7species.mc9nr. Skipping this module.

2026-06-24 16:37:00,725 - pyscenic.transform - WARNING - Less than 80% of the genes in Regulon for ENO1 could be mapped to hg19-500bp-upstrea


2026-06-24 16:37:35,733 - pyscenic.transform - WARNING - Less than 80% of the genes in MZF1 could be mapped to hg19-500bp-upstream-7species.mc9nr. Skipping this module.

2026-06-24 16:37:36,072 - pyscenic.transform - WARNING - Less than 80% of the genes in NFIL3 could be mapped to hg19-500bp-upstream-7species.mc9nr. Skipping this module.

2026-06-24 16:37:36,605 - pyscenic.transform - WARNING - Less than 80% of the genes in NFKB2 could be mapped to hg19-500bp-upstream-7species.mc9nr. Skipping this module.

2026-06-24 16:37:37,211 - pyscenic.transform - WARNING - Less than 80% of the genes in MYPOP could be mapped to hg19-tss-centered-5kb-7species.mc9nr. Skipping this module.

2026-06-24 16:37:37,236 - pyscenic.transform - WARNING - Less than 80% of the genes in MZF1 could be mapped to hg19-tss-centered-5kb-7species.mc9nr. Skipping this module.

2026-06-24 16:37:39,329 - pyscenic.transform - WARNING - Less than 80% of the genes in Regulon for KMT2A could be mapped to hg19-tss-centered-


2026-06-24 16:38:25,887 - pyscenic.transform - WARNING - Less than 80% of the genes in Regulon for RNF138 could be mapped to hg19-tss-centered-5kb-7species.mc9nr. Skipping this module.

2026-06-24 16:38:27,766 - pyscenic.transform - WARNING - Less than 80% of the genes in YY1 could be mapped to hg19-500bp-upstream-7species.mc9nr. Skipping this module.

2026-06-24 16:38:28,962 - pyscenic.transform - WARNING - Less than 80% of the genes in ZBTB25 could be mapped to hg19-500bp-upstream-7species.mc9nr. Skipping this module.

2026-06-24 16:38:29,067 - pyscenic.transform - WARNING - Less than 80% of the genes in ZBTB39 could be mapped to hg19-500bp-upstream-7species.mc9nr. Skipping this module.

2026-06-24 16:38:29,653 - pyscenic.transform - WARNING - Less than 80% of the genes in ZBTB40 could be mapped to hg19-500bp-upstream-7species.mc9nr. Skipping this module.

2026-06-24 16:38:29,897 - pyscenic.transform - WARNING - Less than 80% of the genes in ZBTB6 could be mapped to hg19-500bp-upstr


2026-06-24 16:38:49,795 - pyscenic.transform - WARNING - Less than 80% of the genes in ZNF304 could be mapped to hg19-tss-centered-5kb-7species.mc9nr. Skipping this module.

2026-06-24 16:38:50,584 - pyscenic.transform - WARNING - Less than 80% of the genes in Regulon for ZCCHC17 could be mapped to hg19-500bp-upstream-7species.mc9nr. Skipping this module.

2026-06-24 16:38:50,843 - pyscenic.transform - WARNING - Less than 80% of the genes in ZNF660 could be mapped to hg19-500bp-upstream-7species.mc9nr. Skipping this module.

2026-06-24 16:38:50,873 - pyscenic.transform - WARNING - Less than 80% of the genes in ZNF324 could be mapped to hg19-tss-centered-5kb-7species.mc9nr. Skipping this module.

2026-06-24 16:38:50,892 - pyscenic.transform - WARNING - Less than 80% of the genes in Regulon for ZFP3 could be mapped to hg19-500bp-upstream-7species.mc9nr. Skipping this module.

2026-06-24 16:38:51,192 - pyscenic.transform - WARNING - Less than 80% of the genes in ZNF688 could be mapped to

...


2026-06-24 16:43:44,756 - pyscenic.transform - WARNING - Less than 80% of the genes in Regulon for KMT2A could be mapped to hg19-tss-centered-5kb-7species.mc9nr. Skipping this module.

2026-06-24 16:43:46,471 - pyscenic.transform - WARNING - Less than 80% of the genes in Regulon for SMAD5 could be mapped to hg19-500bp-upstream-7species.mc9nr. Skipping this module.

2026-06-24 16:43:47,728 - pyscenic.transform - WARNING - Less than 80% of the genes in Regulon for SNAPC4 could be mapped to hg19-500bp-upstream-7species.mc9nr. Skipping this module.

2026-06-24 16:43:49,947 - pyscenic.transform - WARNING - Less than 80% of the genes in Regulon for SP110 could be mapped to hg19-500bp-upstream-7species.mc9nr. Skipping this module.

2026-06-24 16:43:56,124 - pyscenic.transform - WARNING - Less than 80% of the genes in Regulon for MORN1 could be mapped to hg19-tss-centered-5kb-7species.mc9nr. Skipping this module.

2026-06-24 16:43:58,598 - pyscenic.transform - WARNING - Less than 80% of the g


2026-06-24 16:44:52,628 - pyscenic.transform - WARNING - Less than 80% of the genes in Regulon for ZNF628 could be mapped to hg19-500bp-upstream-7species.mc9nr. Skipping this module.

2026-06-24 16:44:52,683 - pyscenic.transform - WARNING - Less than 80% of the genes in Regulon for ZNF630 could be mapped to hg19-500bp-upstream-7species.mc9nr. Skipping this module.

2026-06-24 16:44:52,869 - pyscenic.transform - WARNING - Less than 80% of the genes in Regulon for ZNF653 could be mapped to hg19-500bp-upstream-7species.mc9nr. Skipping this module.

2026-06-24 16:44:53,804 - pyscenic.transform - WARNING - Less than 80% of the genes in Regulon for ZNF66 could be mapped to hg19-500bp-upstream-7species.mc9nr. Skipping this module.

2026-06-24 16:44:54,312 - pyscenic.transform - WARNING - Less than 80% of the genes in Regulon for ZNF688 could be mapped to hg19-500bp-upstream-7species.mc9nr. Skipping this module.

2026-06-24 16:44:56,138 - pyscenic.transform - WARNING - Less than 80% of the ge


2026-06-24 16:45:37,926 - pyscenic.prune - INFO - Worker hg19-500bp-upstream-7species.mc9nr(2): All regulons derived.

2026-06-24 16:45:37,926 - pyscenic.prune - INFO - Worker hg19-500bp-upstream-7species.mc9nr(2): All regulons derived.

2026-06-24 16:45:38,000 - pyscenic.prune - INFO - Worker hg19-500bp-upstream-7species.mc9nr(2): Done.

2026-06-24 16:45:38,000 - pyscenic.prune - INFO - Worker hg19-500bp-upstream-7species.mc9nr(2): Done.

2026-06-24 16:45:42,853 - pyscenic.transform - WARNING - Less than 80% of the genes in Regulon for ZNF576 could be mapped to hg19-tss-centered-5kb-7species.mc9nr. Skipping this module.

2026-06-24 16:45:42,967 - pyscenic.transform - WARNING - Less than 80% of the genes in Regulon for ZNF582 could be mapped to hg19-tss-centered-5kb-7species.mc9nr. Skipping this module.

2026-06-24 16:45:43,036 - pyscenic.transform - WARNING - Less than 80% of the genes in Regulon for ZNF585A could be mapped to hg19-tss-centered-5kb-7species.mc9nr. Skipping this modul

Create regulons from a dataframe of enriched features.
Additional columns saved: []


100%|████████████████████████████████████████| 419/419 [00:00<00:00, 830.29it/s]


In [63]:
import pickle
import pandas as pd

with open(REGULONS_FNAME, 'rb') as f:
    regulons = pickle.load(f)

edges = []

for reg in regulons:
    tf = reg.transcription_factor
    for target, weight in reg.gene2weight.items():
        edges.append([tf, target, weight])

df = pd.DataFrame(edges, columns=["source", "target", "weight"])
df.to_csv("/home/melanie/Documents/Projects/CLL_GRN_conditions/Results/GRN_CLL_WO_pruned.csv", index=False)


# Phase III: Cellular regulon enrichment matrix (aka AUCell)

Characterize the different cells in a single-cell transcriptomics experiment by the enrichment of the regulons. Enrichment of a regulon is measures as AUC of the recovery curve of the genes that define this regulon.

In [52]:
auc_mtx = aucell(ex_matrix, regulons, num_workers=1)

100%|████████████████████████████████████████| 432/432 [00:00<00:00, 739.00it/s]


In [19]:
sns.clustermap(auc_mtx, figsize=(12,12))

NameError: name 'sns' is not defined

In [53]:
auc_mtx.to_csv(AUCell_FNAME, index=True, sep='\t')